# Basic KV Cache Correctness & Performance benchmark:

This notebook implements a basic benchmark for KVCache that shows,
- Correctness : no-cache vs KV-cache token equivalence under fixed seed
- Performance metrics : end-to-end latency, decode latency (ms/token), throughput (tokens/sec) 

In [1]:
!nvidia-smi
!nvidia-smi --query-gpu=memory.total,memory.used
!lscpu | grep cache
!free -h

In [2]:
!wget -r -np -nH --reject "index.html*" -X "/.git" -P remote https://hirundine-hipolito-progravid.ngrok-free.dev

In [3]:
pip install torch transformers matplotlib

In [5]:
import sys
import os
import torch
import time
import random
import matplotlib.pyplot as plt
from transformers import GPT2LMHeadModel 
from transformers import GPT2Tokenizer

# Add the parent directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))  # Adjust the path as needed

from model.gpt import GPT, GPTConfig, load_hf_weights
from inference.generate import generate

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

def set_seed(seed: int = 1234):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


In [3]:
# Load HF pretrained weights
hf_model = GPT2LMHeadModel.from_pretrained("gpt2")
hf_model.eval()

# Match the configuration of the pre-trained GPT-2 model
config = GPTConfig(
	vocab_size=50257,  # Match GPT-2 vocab size
	n_embd=768,        # Match GPT-2 embedding size
	n_layer=12,        # Match GPT-2 number of layers
	n_head=12         # Match GPT-2 number of attention heads
)
model = GPT(config).to(device)
load_hf_weights(model, hf_model)
model.eval()


In [ ]:
seed = 400

tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
prompt = "Once upon a time"
max_context_len = 1024
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
generated_tokens = max_context_len - input_ids.shape[1]

if generated_tokens < 0:
    raise ValueError(
        f"Prompt length {input_ids.shape[1]} exceeds max_context_len={max_context_len}"
    )

set_seed(seed)
start_time_1 = time.time()
output_no_cache = generate(model, input_ids.clone(), use_kv_cache=False, max_context_len=max_context_len)
end_time_1 = time.time()

set_seed(seed)
start_time_2 = time.time()
output_cache = generate(model, input_ids.clone(), use_kv_cache=True, max_context_len=max_context_len)
end_time_2 = time.time()

match = torch.equal(output_no_cache, output_cache)
print(f"Outputs match: {match}")

print("Without KV cache:")
print("Output:", tokenizer.decode(output_no_cache[0], skip_special_tokens=True))
print(f"End-to-end latency: {end_time_1 - start_time_1:.2f} seconds")
print(f"Generated tokens: {generated_tokens}")
print(f"Decode ms per token: {(end_time_1 - start_time_1) / generated_tokens * 1000:.2f} ms/token")
print(f"Throughput: {generated_tokens / (end_time_1 - start_time_1):.2f} tokens/sec\n")


print("With KV cache:")
print("Output:", tokenizer.decode(output_cache[0], skip_special_tokens=True))
print(f"End-to-end latency: {end_time_2 - start_time_2:.2f} seconds")
print(f"Generated tokens: {generated_tokens}")
print(f"Decode ms per token: {(end_time_2 - start_time_2) / generated_tokens * 1000:.2f} ms/token")
print(f"Throughput: {generated_tokens / (end_time_2 - start_time_2):.2f} tokens/sec")


In [ ]:
labels = ["Without KV cache", "With KV cache"]
latencies = [end_time_1 - start_time_1, end_time_2 - start_time_2]
decode_ms = [latency / generated_tokens * 1000 for latency in latencies]
throughputs = [generated_tokens / latency for latency in latencies]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
colors = ["#d95f02", "#1b9e77"]
metrics = [
    (latencies, "End-to-end latency", "Seconds"),
    (decode_ms, "Decode latency", "ms/token"),
    (throughputs, "Throughput", "tokens/sec"),
]

for ax, (values, title, ylabel) in zip(axes, metrics):
    bars = ax.bar(labels, values, color=colors)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=10)
    for bar, value in zip(bars, values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:.2f}",
            ha="center",
            va="bottom",
        )

fig.suptitle(f"KV Cache Performance Comparison (total context={max_context_len})")
fig.tight_layout()
plt.show()
